# Kenya Food Prices Capstone Storytelling Notebook

This notebook is designed for presentation and demo support. It explains the project outputs from raw CSV profiling through cleaning, quality checks, SQL-style analysis, and pipeline readiness.

It is intentionally local-first, matching the capstone brief and the current repo setup.

## Verified Run Notes From This Environment

- `python/extract.py` ran successfully and loaded `18,837` rows with `16` raw columns.
- `python/clean.py wfp_food_prices_ken.csv` ran successfully and produced `18,837` cleaned rows with derived columns such as `year`, `month`, and `price_per_kg`.
- Quality checks passed on the current CSV, with `0` invalid rows flagged.
- `python/pipeline.py --local wfp_food_prices_ken.csv --skip-snowflake` started successfully but stopped at the PostgreSQL load step because `PG_HOST` was not set.
- Docker-backed services could not be run here because the Docker daemon was not active, so PostgreSQL, Airflow, and dbt remain pending service startup.

In [79]:
from pathlib import Path
import sys
import subprocess

import pandas as pd
from dotenv import load_dotenv

ROOT = Path.cwd()
if not (ROOT / 'python').exists():
    ROOT = ROOT.parent

load_dotenv(ROOT / '.env')

sys.path.insert(0, str(ROOT / 'python'))

from extract import extract
from clean import clean
from quality import run_checks

DATA_PATH = ROOT / 'wfp_food_prices_ken.csv'
raw_df = extract(str(DATA_PATH))
clean_df = clean(raw_df)
quality_results = run_checks(clean_df)

print(f'Project root: {ROOT}')
print(f'Data path: {DATA_PATH}')
print(f'Raw shape: {raw_df.shape}')
print(f'Clean shape: {clean_df.shape}')

2026-04-04 23:19:45 | INFO | extract | Loading data from local path: /Users/leonida/Documents/code/kenya_food_prices/wfp_food_prices_ken.csv


2026-04-04 23:19:46 | INFO | extract | Loaded 18837 rows from /Users/leonida/Documents/code/kenya_food_prices/wfp_food_prices_ken.csv
2026-04-04 23:19:46 | INFO | clean | Starting clean step — input shape: (18837, 16)
2026-04-04 23:19:50 | INFO | clean | Flagged 0 rows as invalid (is_valid_price=False).
2026-04-04 23:19:50 | INFO | clean | Clean step complete — output shape: (18837, 20)
2026-04-04 23:19:50 | INFO | quality | =======================================================
2026-04-04 23:19:50 | INFO | quality | DATA QUALITY REPORT
2026-04-04 23:19:50 | INFO | quality | =======================================================
2026-04-04 23:19:50 | INFO | quality | ✅ PASS | not_empty                 | DataFrame has 18837 rows.
2026-04-04 23:19:50 | INFO | quality | ✅ PASS | required_columns          | All required columns present.
2026-04-04 23:19:50 | INFO | quality | ✅ PASS | price_nulls               | 0.0% of price_kes values are null.
2026-04-04 23:19:50 | INFO | quality | ✅ P

Project root: /Users/leonida/Documents/code/kenya_food_prices
Data path: /Users/leonida/Documents/code/kenya_food_prices/wfp_food_prices_ken.csv
Raw shape: (18837, 16)
Clean shape: (18837, 20)


## 1. Raw Dataset Snapshot

The project starts from the downloaded WFP Kenya food prices CSV. This is the month 1 foundation: understanding the raw source before building warehouse layers.

In [80]:
snapshot = pd.DataFrame(
    {
        'metric': [
            'rows',
            'columns',
            'date_min',
            'date_max',
            'markets',
            'commodities',
            'distinct_units',
            'missing_admin1',
            'missing_admin2',
        ],
        'value': [
            len(raw_df),
            raw_df.shape[1],
            raw_df['date'].min(),
            raw_df['date'].max(),
            raw_df['market'].nunique(),
            raw_df['commodity'].nunique(),
            raw_df['unit'].nunique(),
            int(raw_df['admin1'].isna().sum()),
            int(raw_df['admin2'].isna().sum()),
        ],
    }
)

snapshot

,metric,value
0,rows,18837
1,columns,16
2,date_min,2006-01-15
3,date_max,2026-03-15
4,markets,226
5,commodities,51
6,distinct_units,14
7,missing_admin1,63
8,missing_admin2,63


In [81]:
raw_df.head(10)

,date,admin1,admin2,market,market_id,latitude,longitude,category,commodity,commodity_id,unit,priceflag,pricetype,currency,price,usdprice
0,2006-01-15,Coast,Mombasa,Mombasa,191,-4.05,39.67,cereals and tubers,Maize,51,KG,actual,Wholesale,KES,16.13,0.22
1,2006-01-15,Coast,Mombasa,Mombasa,191,-4.05,39.67,cereals and tubers,Maize (white),67,90 KG,actual,Wholesale,KES,1480.00,20.58
2,2006-01-15,Coast,Mombasa,Mombasa,191,-4.05,39.67,pulses and nuts,Beans,50,KG,actual,Wholesale,KES,33.63,0.47
3,2006-01-15,Coast,Mombasa,Mombasa,191,-4.05,39.67,pulses and nuts,Beans (dry),262,90 KG,actual,Wholesale,KES,3246.00,45.15
4,2006-01-15,Eastern,Kitui,Kitui,187,-1.37,38.02,cereals and tubers,Maize (white),67,KG,actual,Retail,KES,17.00,0.24
5,2006-01-15,Eastern,Kitui,Kitui,187,-1.37,38.02,cereals and tubers,Potatoes (Irish),148,50 KG,actual,Wholesale,KES,1249.99,17.39
6,2006-01-15,Eastern,Marsabit,Marsabit,190,2.33,37.98,cereals and tubers,Maize (white),67,KG,actual,Retail,KES,21.00,0.29
7,2006-01-15,Nairobi,Nairobi,Nairobi,184,-1.28,36.82,cereals and tubers,Bread,55,400 G,actual,Retail,KES,26.00,0.36
8,2006-01-15,Nairobi,Nairobi,Nairobi,184,-1.28,36.82,cereals and tubers,Potatoes (Irish),148,50 KG,actual,Wholesale,KES,664.43,9.24
9,2006-01-15,Nairobi,Nairobi,Nairobi,184,-1.28,36.82,cereals and tubers,Sorghum,65,90 KG,actual,Wholesale,KES,1960.00,27.26


## 2. Cleaning And Data Quality Outputs

The transformation layer standardizes column names, parses dates, preserves source identifiers, derives `year`, `month`, and `price_per_kg`, and flags invalid records without deleting them.

For this specific CSV, the source data is relatively clean, which is a useful finding in itself.

In [82]:
quality_summary = pd.DataFrame(
    [
        {
            'check': result.name,
            'passed': result.passed,
            'level': result.level,
            'detail': result.detail,
        }
        for result in quality_results
    ]
)

quality_summary

,check,passed,level,detail
0,not_empty,True,critical,DataFrame has 18837 rows.
1,required_columns,True,critical,All required columns present.
2,price_nulls,True,warning,0.0% of price_kes values are null.
3,date_range,True,warning,Date range: 2006-01-15 → 2026-03-15.
4,negative_prices,True,critical,0 negative price values found.
5,duplicate_keys,True,warning,0 duplicate key rows found.
6,valid_price_ratio,True,warning,100.0% of rows have valid prices.


In [83]:
issues = {
    'rows_with_missing_county': int(clean_df['county'].isna().sum()),
    'rows_with_missing_district': int(clean_df['district'].isna().sum()),
    'rows_with_missing_price_kes': int(clean_df['price_kes'].isna().sum()),
    'rows_flagged_invalid': int((~clean_df['is_valid_price']).sum()),
    'duplicate_business_keys': int(clean_df.duplicated(subset=['date', 'market', 'commodity', 'unit', 'pricetype']).sum()),
}

pd.DataFrame({'issue': list(issues.keys()), 'value': list(issues.values())})

,issue,value
0,rows_with_missing_county,63
1,rows_with_missing_district,63
2,rows_with_missing_price_kes,0
3,rows_flagged_invalid,0
4,duplicate_business_keys,0


In [84]:
unit_examples = (
    clean_df.groupby('commodity')['unit']
    .nunique()
    .reset_index(name='distinct_units_for_commodity')
    .sort_values('distinct_units_for_commodity', ascending=False)
    .head(10)
)

unit_examples

,commodity,distinct_units_for_commodity
18,Kale,3
3,Beans (Dry),2
9,Cabbage,2
37,Potatoes (Irish),2
41,Rice (Aromatic),2
49,Tomatoes,2
44,Sorghum,2
30,Milk (Uht),2
29,"Milk (Cow, Pasteurized)",2
21,"Maize (White, Dry)",2


## 3. SQL-Style Analytical Outputs

These tables mirror the kinds of deliverables requested in month 1: filtering, sorting, aggregates, grouping, date handling, and trend analysis. Here they are shown in pandas so the storytelling notebook remains runnable even before PostgreSQL is started.

Important interpretation note: when comparing across counties or across time, this notebook prefers `price_per_kg` where possible. That avoids mixing prices from `KG`, `90 KG`, `50 KG`, and other unit types in the same average.

In [85]:
clean_df['date'] = pd.to_datetime(clean_df['date'])

latest_nairobi = (
    clean_df.loc[clean_df['market'].str.contains('nairobi', case=False, na=False), ['date', 'market', 'commodity', 'unit', 'price_kes', 'currency']]
    .sort_values(['date', 'commodity'], ascending=[False, True])
    .head(15)
)

latest_nairobi

,date,market,commodity,unit,price_kes,currency
12512,2026-02-15,Kangemi (Nairobi),Beans (Dolichos),90 KG,10250.10,KES
12513,2026-02-15,Kangemi (Nairobi),Beans (Rosecoco),90 KG,11600.00,KES
12514,2026-02-15,Kangemi (Nairobi),Cowpeas,90 KG,9800.00,KES
12510,2026-02-15,Kangemi (Nairobi),"Maize (White, Dry)",90 KG,4599.90,KES
12511,2026-02-15,Kangemi (Nairobi),Rice (Aromatic),50 KG,7250.00,KES
12515,2026-02-15,Kangemi (Nairobi),Tomatoes,64 KG,4430.72,KES
12507,2026-01-15,Kangemi (Nairobi),Rice (Aromatic),50 KG,7875.00,KES
12290,2025-10-15,Kangemi (Nairobi),Beans (Dolichos),90 KG,5700.00,KES
12288,2025-10-15,Kangemi (Nairobi),"Maize (White, Dry)",90 KG,4175.00,KES
12291,2025-10-15,Kangemi (Nairobi),Onions (Dry),13 KG,1040.00,KES


In [86]:
commodity_stats = (
    clean_df.dropna(subset=['price_per_kg'])
    .groupby('commodity')['price_per_kg']
    .agg(avg_price_per_kg='mean', min_price_per_kg='min', max_price_per_kg='max', observations='count')
    .reset_index()
    .sort_values('avg_price_per_kg', ascending=False)
    .head(15)
)

commodity_stats.round(2)

,commodity,avg_price_per_kg,min_price_per_kg,max_price_per_kg,observations
20,Meat (Goat),612.14,400.00,847.69,301
19,Meat (Camel),534.49,400.00,743.52,126
18,Meat (Beef),534.02,5.00,800.00,276
23,Pigeon Peas (Dry),209.54,87.00,700.00,97
7,Cooking Fat,188.20,104.00,380.00,113
10,Cowpeas (Dry),161.48,50.00,400.00,206
34,Sugar,158.87,90.36,300.00,882
11,"Fish (Omena, Dry)",154.56,97.00,200.00,18
28,Rice (Aromatic),139.58,75.00,245.00,919
27,Rice,119.81,73.33,220.00,521


In [87]:
avg_price_by_county_year = (
    clean_df.dropna(subset=['price_per_kg'])
    .groupby(['county', 'year'], dropna=False)['price_per_kg']
    .agg(avg_price_per_kg='mean', records='count')
    .reset_index()
)

avg_price_by_county_year.query('records >= 5').sort_values(['year', 'avg_price_per_kg'], ascending=[False, False]).head(20).round(2)

,county,year,avg_price_per_kg,records
85,North Eastern,2026,228.47,45
64,Nairobi,2026,117.55,6
126,Rift Valley,2025,184.08,741
84,North Eastern,2025,162.44,348
23,Coast,2025,118.38,73
43,Eastern,2025,102.89,72
63,Nairobi,2025,94.28,34
105,Nyanza,2025,57.02,21
125,Rift Valley,2024,150.15,1729
83,North Eastern,2024,148.42,722


In [88]:
maize_trend = (
    clean_df.loc[
        clean_df['commodity'].str.contains('maize', case=False, na=False)
        & clean_df['price_per_kg'].notna()
    ]
    .groupby('year')['price_per_kg']
    .agg(avg_price_per_kg='mean', observations='count')
    .reset_index()
    .sort_values('year')
)

maize_trend['prev_year_price_per_kg'] = maize_trend['avg_price_per_kg'].shift(1)
maize_trend['yoy_pct_change'] = ((maize_trend['avg_price_per_kg'] - maize_trend['prev_year_price_per_kg']) / maize_trend['prev_year_price_per_kg']) * 100

maize_trend.round(2).tail(12)

,year,avg_price_per_kg,observations,prev_year_price_per_kg,yoy_pct_change
9,2015,40.22,172,40.01,0.53
10,2016,40.15,166,40.22,-0.19
11,2017,51.02,181,40.15,27.08
12,2018,40.39,170,51.02,-20.83
13,2019,45.36,156,40.39,12.29
14,2020,49.26,108,45.36,8.60
15,2021,50.18,482,49.26,1.86
16,2022,55.18,141,50.18,9.96
17,2023,91.05,357,55.18,65.02
18,2024,83.67,758,91.05,-8.11


## 4. Pipeline Execution Status In This Environment

The local Python modules can run immediately from the repo. The database-backed steps depend on PostgreSQL, Airflow, and dbt being available.

The cells below document the actual run state from this machine so your presentation can clearly separate verified outputs from environment blockers.

In [89]:
import os
import re

ANSI_RE = re.compile(r'\x1b\[[0-9;]*m')

def clean_text(text):
    return ANSI_RE.sub('', text or '').strip()

docker_check = subprocess.run(
    ['docker', 'ps', '--format', '{{.Names}}'],
    cwd=ROOT,
    capture_output=True,
    text=True,
)

required_pg_env = ['PG_HOST', 'PG_DATABASE', 'PG_USER', 'PG_PASSWORD']
missing_pg_env = [name for name in required_pg_env if not os.environ.get(name)]

if missing_pg_env:
    pipeline_stdout = ''
    pipeline_stderr = f"Pipeline not executed from notebook because required environment variables are missing: {', '.join(missing_pg_env)}"
    pipeline_return_code = 1
else:
    pipeline_check = subprocess.run(
        ['python3', 'python/pipeline.py', '--local', 'wfp_food_prices_ken.csv', '--skip-snowflake'],
        cwd=ROOT,
        capture_output=True,
        text=True,
    )
    pipeline_stdout = clean_text(pipeline_check.stdout)
    pipeline_stderr = clean_text(pipeline_check.stderr)
    pipeline_return_code = pipeline_check.returncode

status_df = pd.DataFrame(
    [
        {
            'component': 'docker daemon',
            'return_code': docker_check.returncode,
            'status': 'ready' if docker_check.returncode == 0 else 'blocked',
            'detail': clean_text(docker_check.stderr or docker_check.stdout)[:220],
        },
        {
            'component': 'python pipeline',
            'return_code': pipeline_return_code,
            'status': 'ready' if pipeline_return_code == 0 else 'blocked',
            'detail': (pipeline_stderr or pipeline_stdout).splitlines()[-1][:220] if (pipeline_stderr or pipeline_stdout) else '',
        },
    ]
)

status_df

,component,return_code,status,detail
0,docker daemon,0,ready,kfp_airflow_web\nkfp_airflow_scheduler\nkfp_gr...
1,python pipeline,0,ready,2026-04-04 23:20:56 | INFO | pipeline | ======...


In [90]:
print('Pipeline stdout tail:')
print('\n'.join(pipeline_stdout.splitlines()[-12:]))
if pipeline_stderr:
    print('\nPipeline stderr tail:')
    print('\n'.join(pipeline_stderr.splitlines()[-12:]))

Pipeline stdout tail:


Pipeline stderr tail:
2026-04-04 23:20:53 | INFO | quality | ✅ PASS | negative_prices           | 0 negative price values found.
2026-04-04 23:20:53 | INFO | quality | ✅ PASS | duplicate_keys            | 0 duplicate key rows found.
2026-04-04 23:20:53 | INFO | quality | ✅ PASS | valid_price_ratio         | 100.0% of rows have valid prices.
2026-04-04 23:20:53 | INFO | quality | =======================================================
2026-04-04 23:20:53 | INFO | pipeline | [5/5] LOAD (staging → dimensions → fact)
2026-04-04 23:20:54 | INFO | load | Incremental filter kept 0 of 18837 rows.
2026-04-04 23:20:54 | INFO | load | No new data to load (incremental check).
2026-04-04 23:20:55 | INFO | load | Dimensions populated (warehouse.dim_market, warehouse.dim_commodity).
2026-04-04 23:20:56 | INFO | load | Fact table populated.
2026-04-04 23:20:56 | INFO | pipeline | ============================================================
2026-04-04 23:20:56 | INFO | pipeline 

## 5. Deliverable Readiness Map

This is a practical slide-friendly summary of what is already validated locally and what still needs the service layer to be running.

In [91]:
deliverables = pd.DataFrame(
    [
        ['Raw CSV profiling', 'Verified locally', 'Notebook and Python modules ran on the local CSV'],
        ['Cleaning logic', 'Verified locally', 'clean.py produced expected derived columns and no invalid rows for this CSV'],
        ['Quality checks', 'Verified locally', 'All quality checks passed on the current dataset'],
        ['SQL query logic', 'Verified analytically', 'Notebook reproduces the intended SQL outputs in pandas'],
        ['PostgreSQL load', 'Pending services', 'Requires PostgreSQL plus PG_* environment variables'],
        ['Airflow DAG run', 'Pending services', 'Requires Docker daemon and Airflow containers'],
        ['dbt run/test', 'Pending services', 'Requires dbt installation and a running warehouse target'],
        ['Snowflake mirror', 'Optional and pending credentials', 'Only needed if you want the dual-target demo'],
    ],
    columns=['deliverable', 'status', 'notes'],
)

deliverables

,deliverable,status,notes
0,Raw CSV profiling,Verified locally,Notebook and Python modules ran on the local CSV
1,Cleaning logic,Verified locally,clean.py produced expected derived columns and...
2,Quality checks,Verified locally,All quality checks passed on the current dataset
3,SQL query logic,Verified analytically,Notebook reproduces the intended SQL outputs i...
4,PostgreSQL load,Pending services,Requires PostgreSQL plus PG_* environment vari...
5,Airflow DAG run,Pending services,Requires Docker daemon and Airflow containers
6,dbt run/test,Pending services,Requires dbt installation and a running wareho...
7,Snowflake mirror,Optional and pending credentials,Only needed if you want the dual-target demo


## Presentation Close

A strong way to narrate this capstone is:

1. Start with the Kenya food prices problem and the raw WFP dataset.
2. Show that you profiled the source carefully before modeling it.
3. Explain how cleaning preserves lineage while making analysis easier.
4. Use the SQL-style outputs to show real analytical value.
5. Close by distinguishing what is already validated locally from what needs the Docker/PostgreSQL stack to be running for a full warehouse demo.

That framing makes the project feel disciplined, honest, and production-minded.